## Descarga de ortofoto y Modelo Digital de terreno de los servidores del IGN

In [ ]:
from owslib.wms import WebMapService
from pyproj import Transformer
from shapely import Point 
from owslib.wcs import WebCoverageService



import os
import re
import math
location = "fuego14"
def dm2dd(coord_str):
    deg, minutes, seconds, direction =  re.split('[°\'"]', coord_str)
    return (abs(float(deg)) + float(minutes)/60 + float(seconds)/(60*60)) * (-1 if direction in ['W', 'S'] else 1)



locations = {
    "tarifa": {"lat": dm2dd('''36°7'9"N'''),"lng": dm2dd('''5°49'52"W''')},
    "tarifa2": {"lat": dm2dd('''36°7'37"N'''),"lng": dm2dd('''5°50'16"W''')},
    "losacio": {"lat": dm2dd('''41°44'53.268"N'''),"lng": dm2dd('''-5°55'35.088"W''')},
    "uleila": {"lat": dm2dd('''37°12'5.778"N'''),"lng": dm2dd('''-2°11'12.456"W''')},
    "pizarra": {"lat": dm2dd('''36°46'42.378"N'''),"lng": dm2dd('''4°43'29.868"W''')},
    "cabeza": {"lat": dm2dd('''38°34'55"N'''),"lng": dm2dd('''5°20'16"W''')},
    "carracedelo": {"x": 683907,"y": 4687122, "T":29},
    "teruel":  {"x": 633262,"y": 4549637, "T":30},
    "fuego01": {"x": 358787,"y": 4717905, "T":30},
    "fuego02": {"x": 360492,"y": 4717682, "T":30},
    "fuego03": {"x": 360957,"y": 4719131, "T":30},
    "fuego04": {"x": 360709,"y": 4719161, "T":30},
    "fuego05": {"x": 272501,"y": 4604315, "T":30},
    "fuego06": {"x": 259561,"y": 4721527, "T":30},
    "fuego07": {"x": 680017,"y": 4721557, "T":29},
    "fuego08": {"x": 683423,"y": 4700934, "T":29},
    "fuego09": {"x": 738469,"y": 4670119, "T":29},
    "fuego10": {"x": 341876,"y": 4608538, "T":30},
    "fuego11": {"x": 279560,"y": 4745775, "T":30},
    "fuego12": {"x": 666268,"y": 4656622, "T":29},
    "fuego13": {"x": 279423,"y": 4746057, "T":30},
    "fuego14": {"x": 385159,"y": 4493156, "T":30},
    "fuego17": {"x": 454299,"y": 4706205, "T":30},
    "fuego19": {"x": 337191,"y": 4198190, "T":30},
    "fuego20": {"lat": dm2dd('''37°58'32"N'''),"lng": dm2dd('''6°16'4"W''')},
    "fuego21": {"lat": dm2dd('''38°6'27"N'''),"lng": dm2dd('''3°9'52"W''')},
    "fuego22": {"lat": dm2dd('''37°31'11"N'''),"lng": dm2dd('''6°24'31"W''')},
    "fuego23": {"lat": dm2dd('''36°57'12"N'''),"lng": dm2dd('''3°7'41"W''')},
    "fuego24": {"x": 725262,"y": 4731635, "T":29},

}

print(locations[location])
l = locations[location]
scaler = 1
image_size = 2000 * scaler # Número de pixeles de la imagen, Es una imagen cuadrada por lo que el tamaño de la imagen sera image_sizeximage_size
plane_size = 4 * scaler# Tamaño del escenario en Km 
if not os.path.isdir ( os.path.join( "locations", location ) ):
    print("Creating directory: " + os.path.join("locations", location ) )
    os.mkdir ( os.path.join("locations", location ) )
SOURCE_IMAGE_PATH = os.path.join("images_DB", location + ".jpeg")
TARGET_IMAGE_PATH = os.path.join("locations", location, location + ".jpeg")
command = "copy " + SOURCE_IMAGE_PATH + " " + TARGET_IMAGE_PATH
print(command)
os.system(command)
#os.popen('copy images_DB/{0}.jpeg locations/{0}/{0}.jpeg'.format(location))

orto_path = os.path.join('locations',location,'orto.png') # Ruta donde guardar la imagen aérea 

if ("lat" in l.keys() and "lng" in l.keys()):
    lat = l["lat"]
    lng = l["lng"]
    transformer = Transformer.from_crs(4326, 25830) # Si las coordenadas están en lat / long descomentar esta línea
    point = Point(transformer.transform(lat, lng)) # Transformar coordenadas geográficas a UTM. TODO De momento solo válido para península
    T = 30
else:
    T = l["T"]
    point = Point(l["x"],l["y"]) # Transformar coordenadas geográficas a UTM. TODO De momento solo válido para península

# point = Point(691932, 4182294) # Si las coordenadas están en latitud / longitud comentar esta línea
rect = point.buffer(plane_size * 1000 / 2) # Hay que multiplicar por 1000 para pasar a metros y dividir por dos porque estamos usando el radio respecto al punto de ubicación del dron
print(rect.bounds)
wms = WebMapService('https://www.ign.es/wms-inspire/pnoa-ma', version='1.3.0')
if T == 29:
    srs='EPSG:25829',
else:
    srs='EPSG:25830',
    
img = wms.getmap(   layers=['OI.OrthoimageCoverage'],
                    srs='EPSG:25830',
                    bbox=rect.bounds,
                    size=(image_size, image_size),
                    format='image/png'
                           )
print(point)
with open(orto_path, 'wb') as f:
    f.write(img.read())
    


mdt_path = os.path.join('locations',location,'mdt.tif') # Ruta a guardar imagen del modelo digital del terreno

minx, miny, maxx, maxy = rect.bounds

wcs = WebCoverageService('https://servicios.idee.es/wcs-inspire/mdt', version='2.0.1')

wcs.contents

out = wcs.getCoverage(identifier=[wcs.contents['Elevacion25830_25'].id],
        subsets=[('x', minx, maxx), ('y', miny, maxy)],
        format='GEOTIFFINT16',
        crs='EPSG:25829',
             )

with open(mdt_path, 'wb') as f:
    f.write(out.read())

{'x': 385159, 'y': 4493156, 'T': 30}
cp images_DB\fuego14.jpeg locations\fuego14\fuego14.jpeg
(383159.0, 4491156.0, 387159.0, 4495156.0)
POINT (385159 4493156)


In [ ]:
import os
location = "fuego14"



Creating directory: locations\fuego14
copy images_DB\fuego14.jpeg locations\fuego14\fuego14.jpeg


0

{'Elevacion4258_1000': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936b90>,
 'Elevacion4326_1000': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936410>,
 'Elevacion4258_500': <owslib.coverage.wcs201.ContentMetadata at 0x143c0937790>,
 'Elevacion4326_500': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936710>,
 'Elevacion4258_200': <owslib.coverage.wcs201.ContentMetadata at 0x143c09363b0>,
 'Elevacion4258_25': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936a40>,
 'Elevacion4258_5': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936aa0>,
 'Elevacion25830_1000': <owslib.coverage.wcs201.ContentMetadata at 0x143c09364a0>,
 'Elevacion4083_1000': <owslib.coverage.wcs201.ContentMetadata at 0x143c09361d0>,
 'Elevacion25830_500': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936500>,
 'Elevacion4083_500': <owslib.coverage.wcs201.ContentMetadata at 0x143c0935ff0>,
 'Elevacion25830_200': <owslib.coverage.wcs201.ContentMetadata at 0x143c0936200>,
 'Elevacion4083_200': <o

In [11]:
os.popen('copy images02/{0}.jpeg {0}/{0}.jpeg'.format(location))

In [2]:
from owslib.wms import WebMapService
from pyproj import Transformer
from shapely import Point 
from owslib.wcs import WebCoverageService

def get_N_tiles(point,location):
    neighbours = [[-2,2],[-1,2],[0,2],[1,2],[2,2],
                  [-2,1],[-1,1],[0,1],[1,1],[2,1],
                  [-2,0],[-1,0],[0,0],[1,0],[2,0],
                  [-2,-1],[-1,-1],[0,-1],[1,-1],[2,-1],
                  [-2,-2],[-1,-2],[0,-2],[1,-2],[2,-2]]
    
    #neighbours = [[-1,1],[0,1],[1,1],
    #              [-1,0],[0,0],[1,0],
    #              [-1,-1],[0,-1],[1,-1]]
    idx = 0
    for coords in neighbours:
        #if idx < 5:
        #    idx += 1
        #    continue
        tmp_x = point.x
        tmp_y = point.y
        # orto_path = os.path.join(location,"orto_"+idx+".png") # Ruta donde guardar la imagen aérea 
        
        tmp_x += coords[0] * plane_size * 1000 
        tmp_y += coords[1] * plane_size * 1000 
        tempPnt = Point(tmp_x,tmp_y)
        rect = tempPnt.buffer(plane_size * 1000 / 2) # Hay que multiplicar por 1000 para pasar a metros y dividir por dos porque estamos usando el radio respecto al punto de ubicación del dron
        wms = WebMapService('https://www.ign.es/wms-inspire/pnoa-ma', version='1.3.0')
        if location == "carracedelo":
            srs='EPSG:25829',
        else:
            srs='EPSG:25830',
            
        img = wms.getmap(   layers=['OI.OrthoimageCoverage'],
                            srs='EPSG:25830',
                            bbox=rect.bounds,
                            size=(image_size, image_size),
                            format='image/png'
                                )
        orto_path = os.path.join(location,"orto_"+str(idx)+".png") # Ruta donde guardar la imagen aérea 

        with open(orto_path, 'wb') as f:
            f.write(img.read())

        mdt_path = os.path.join(location,"mdt_"+str(idx)+".tif") # Ruta a guardar imagen del modelo digital del terreno

        minx, miny, maxx, maxy = rect.bounds

        wcs = WebCoverageService('https://servicios.idee.es/wcs-inspire/mdt', version='2.0.1')

        wcs.contents

        out = wcs.getCoverage(identifier=[wcs.contents['Elevacion25830_25'].id],
                subsets=[('x', minx, maxx), ('y', miny, maxy)],
                format='GEOTIFFINT16',
                crs='EPSG:25829',
                    )

        with open(mdt_path, 'wb') as f:
            f.write(out.read())


        idx += 1
get_N_tiles(point,location)

ReadTimeout: HTTPSConnectionPool(host='www.ign.es', port=443): Read timed out. (read timeout=30)

In [ ]:
from owslib.wcs import WebCoverageService

mdt_path = os.path.join(location,'mdt.tif') # Ruta a guardar imagen del modelo digital del terreno

minx, miny, maxx, maxy = rect.bounds

wcs = WebCoverageService('https://servicios.idee.es/wcs-inspire/mdt', version='2.0.1')

wcs.contents

out = wcs.getCoverage(identifier=[wcs.contents['Elevacion25830_25'].id],
        subsets=[('x', minx, maxx), ('y', miny, maxy)],
        format='GEOTIFFINT16',
        crs='EPSG:25829',
             )

with open(mdt_path, 'wb') as f:
    f.write(out.read())

In [10]:
import exifread

with open("images/teruel.jpeg", "rb") as f:
    tags = exifread.process_file(f)

for tag in tags:
    print(tag, tags[tag])

In [67]:
l = dm2dd('''48°53'10.18"N''')
from decimal import Decimal, getcontext
getcontext().prec = 12
f = 48.0 + 53.0/60 + 10.18/3600
d = float("48") + float("53")/60 + float("10")/3600
print(l) 
print(f) 
print(d) 
a = 53/60
b = 10.18/3600
print(a)

48.88616111111111
48.88616111111111
48.88611111111111


0.0028277777777777776

In [70]:
from decimal import Decimal, getcontext

# set precision (number of significant digits)
getcontext().prec = 50

a = Decimal('7') / Decimal('60') + Decimal('9') / Decimal('3600')
print(a)

0.11916666666666666666666666666666666666666666666667


In [29]:
from PIL import Image
from PIL.ExifTags import TAGS

# open the image
image = Image.open("img.jpg")

# extracting the exif metadata
exifdata = image.getexif()

# looping through all the tags present in exifdata
for tagid in exifdata:
    
    # getting the tag name instead of tag id
    tagname = TAGS.get(tagid, tagid)

    # passing the tagid to get its respective value
    value = exifdata.get(tagid)
  
    # printing the final result
    print(f"{tagname:25}: {value}")

ModuleNotFoundError: No module named 'PIL'

## Crear imagen usando blender

In [1]:
import bpy
import math


mdt_path = './mdt.tif' # Ruta a guardar imagen del modelo digital del terreno
orto_path = './orto.png' # Ruta donde guardar la imagen aérea 

### SETTINGS

path = "test.png" # Nombre de fichero de la imagen final
plane_size = 4 # Tamaño del escenario en Km 

### CAMERA SETTINGS

focal_length = 28 # Distancia focal
sensor_width = 36 # Ancho del sensor de la cámar medido en mm
camera_height = 1.0269 # Altitud de la cámara medida en km. Ojo, tiene que ser superior a la altitud del terreno en el punto (0,0)
rotation_euler_x = 65 # Rotación euler sobre eje x medida en grados
rotation_euler_y = 8 # Rotación euler sobre eje y medida en grados
rotation_euler_z = 67 # Rotación euler sobre eje z medida en grados

if "Cube" in bpy.data.meshes: # Borra el cubo inicial
    mesh = bpy.data.meshes["Cube"]
    print("removing mesh", mesh)
    bpy.data.meshes.remove(mesh)

tex = bpy.data.textures.new("mdt", 'IMAGE')
img = bpy.data.images.load(mdt_path)
img.colorspace_settings.name = 'Non-Color'
tex.image = img
tex.extension = 'EXTEND'

bpy.ops.mesh.primitive_plane_add(size=plane_size, location=(0, 0, 0))

bpy.ops.object.mode_set(mode = 'EDIT')
bpy.ops.mesh.subdivide(number_cuts=79)
bpy.ops.mesh.subdivide(number_cuts=1)

bpy.ops.object.mode_set(mode = 'OBJECT')
bpy.ops.object.modifier_add(type='DISPLACE')
bpy.context.object.modifiers['Displace'].texture = tex
bpy.context.object.modifiers['Displace'].texture_coords = 'UV'
bpy.context.object.modifiers['Displace'].strength = 32.768
bpy.context.object.modifiers['Displace'].mid_level = 0

bpy.ops.object.shade_smooth()

obj = bpy.context.object

## Asignar imagen aérea como image texture del material del plano

orto_img = bpy.data.images.load(orto_path)
mat = bpy.data.materials['Material']
mat.use_nodes = True
obj.data.materials.append(mat) 
nodes = mat.node_tree.nodes
node_texture = nodes.new(type='ShaderNodeTexImage')
node_texture.image = orto_img
links = mat.node_tree.links
link = links.new(node_texture.outputs[0], nodes.get("Principled BSDF").inputs[0])
nodes.get("Principled BSDF").inputs[3].default_value = 1 # IOR

## Ajustar posición, angulo y distancia focal de la cámara

camera = bpy.data.objects.get('Camera')
camera.location[0] = 0.01
camera.location[1] = 0
camera.location[2] = camera_height
camera.rotation_euler[0] = rotation_euler_x * math.pi / 180
camera.rotation_euler[1] = rotation_euler_y * math.pi / 180
camera.rotation_euler[2] = rotation_euler_z * math.pi / 180
camera.data.lens = focal_length
camera.data.sensor_width = sensor_width
camera.data.clip_start = 0.01

bpy.context.scene.render.resolution_x = 919
bpy.context.scene.render.resolution_y = 507
bpy.ops.render.render()

bpy.data.images["Render Result"].save_render(filepath=path)

removing mesh <bpy_struct, Mesh("Cube") at 0x49381b0>


TIFFReadDirectory: Warning, Unknown field with tag 33550 (0x830e) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 33922 (0x8482) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34735 (0x87af) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34736 (0x87b0) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34737 (0x87b1) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 33550 (0x830e) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 33922 (0x8482) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34735 (0x87af) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34736 (0x87b0) encountered.
TIFFReadDirectory: Warning, Unknown field with tag 34737 (0x87b1) encountered.


Fra:1 Mem:73.16M (Peak 83.54M) | Time:00:01.43 | Rendering 1 / 64 samples
Fra:1 Mem:73.16M (Peak 83.54M) | Time:00:03.33 | Rendering 25 / 64 samples
Fra:1 Mem:73.16M (Peak 83.54M) | Time:00:05.06 | Rendering 50 / 64 samples
Fra:1 Mem:73.16M (Peak 83.54M) | Time:00:06.01 | Rendering 64 / 64 samples


In [19]:
import importlib
import utils 
import pandas as pd
import geopandas as gpd

importlib.reload(utils)

with open('losacio/contorno_area.txt') as f: # Leer pixeles pasados por Andrés
    img_points = [list(map(lambda x: int(x), line.split(' '))) for line in f.readlines()]

local_coords = list(map(lambda x: utils.image_to_utm(bpy.context.scene, camera, bpy.data.objects.get('Plane'), 919, 507, int(x[0]), int(x[1])), img_points))

UTM_coords = [[point.x + p.x * 1000, point.y + p.y * 1000] for p in local_coords] # point es la variable creada en la primera celda (coordenadas del punto donde se tomó la foto)

df = pd.DataFrame(UTM_coords, columns = ['coord_x', 'coord_y'])
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.coord_x, df.coord_y), crs='EPSG:25830')

In [18]:
gdf.to_file('PRUEBA') # Para guardar el conjunto de puntos ya georreferenciados como shp

In [8]:
## Versión interactiva. Se muestra imagen y se guardan las coordenas de los pixeles en los que se ha clickado

%matplotlib tk
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig = plt.figure(figsize=(20,30))

img=mpimg.imread('./losacio/orig.jpeg')

img_points = []

def onclick(event):
    ix, iy = event.xdata, event.ydata
    img_points.append([ix, iy])
    
cid = fig.canvas.mpl_connect('button_press_event', onclick)

imgplot = plt.imshow(img)

plt.show()

In [4]:
import numpy as np
from PIL import Image
def mem_footprint(path1):
    img = np.array(Image.open(path1))
    mem_bytes = img.nbytes
    memory_mb = mem_bytes /(1024**2) 
    print(memory_mb, "MB in RAM") 

path1 = "images/losacio.jpeg"
path2 = "images/cabeza.jpeg"
mem_footprint(path1)
mem_footprint(path2)
img1 = np.array(Image.open(path1))
img2 = np.array(Image.open(path2))
print(img1.shape)
print(img2.shape)


1.3330450057983398 MB in RAM
0.9922027587890625 MB in RAM
(507, 919, 3)
(510, 680, 3)
